# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/02017711723iot-dotcom/Flyrank_Internship_1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

### Research Question

This project asks whether observable content and search-performance signals can be used to rank pages for human review when prioritizing potential content-refresh opportunities.

The decision supported is the prioritization of content pages for review. The output is a ranked review queue rather than an automatic refresh decision. A human reviewer uses the ranking to investigate page visibility, content age, search performance, search intent, content quality, and other possible explanations before deciding whether to refresh, monitor, investigate further, or take no action.

The main cost of a wrong decision is inefficient editorial effort: a false positive may send attention toward a page that does not require a refresh, while a false negative may delay review of a page showing signs of decline.


In [1]:
# CAPSTONE — SECTION 1: RESEARCH QUESTION

research_question = (
    "Can observable content and search-performance signals be used "
    "to rank pages for human review when prioritizing potential "
    "content-refresh opportunities?"
)

decision_supported = (
    "Help content and SEO teams prioritize which pages should be "
    "reviewed first for a possible refresh."
)

unit_of_analysis = "Content page"

output = "Ranked review score and priority order"

human_action = (
    "A reviewer inspects high-ranked pages and decides whether to "
    "refresh, monitor, investigate further, or take no action."
)

wrong_call_cost = (
    "A false positive can waste editorial effort on a page that "
    "does not need a refresh. A false negative can cause a declining "
    "page to be reviewed later than it should be."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

print("\nUnit of analysis:", unit_of_analysis)
print("Output:", output)
print("Human action:", human_action)
print("Cost of wrong call:", wrong_call_cost)

Research question:
Can observable content and search-performance signals be used to rank pages for human review when prioritizing potential content-refresh opportunities?

Decision supported:
Help content and SEO teams prioritize which pages should be reviewed first for a possible refresh.

Unit of analysis: Content page
Output: Ranked review score and priority order
Human action: A reviewer inspects high-ranked pages and decides whether to refresh, monitor, investigate further, or take no action.
Cost of wrong call: A false positive can waste editorial effort on a page that does not need a refresh. A false negative can cause a declining page to be reviewed later than it should be.


## 2. Data and Target Definition

This section loads the anonymized FlyRank content dataset and defines the prediction target used for the content opportunity scoring task.

The target variable is `is_declining_label`, where a value of `1` indicates that the content's observed trend direction is **Down**, and `0` indicates otherwise.

The dataset is also checked for the required fields used in the analysis, including content age, freshness, visibility, click-through rate, search position, and word count.

The analysis uses the anonymized dataset provided for the internship. No client names, domains, URLs, private queries, or other identifying information are used.

In [5]:
import os
import subprocess
import pandas as pd

REPO_DIR = "/content/Flyrank_Internship_1"

# Clone your repository if it is not already present
if not os.path.exists(REPO_DIR):
    print("Repository not found. Cloning...")
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/02017711723iot-dotcom/Flyrank_Internship_1.git",
            REPO_DIR
        ],
        check=True
    )
else:
    print("Repository already exists.")

# Move into the repository
os.chdir(REPO_DIR)

# Dataset path
DATA_PATH = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

print("Working directory:", os.getcwd())
print("Dataset exists:", os.path.exists(DATA_PATH))

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Repository not found. Cloning...
Working directory: /content/Flyrank_Internship_1
Dataset exists: True
Dataset loaded successfully!
Shape: (30000, 44)


In [6]:
print("Dataset shape:", df.shape)
print("Number of clients:", df["client_id"].nunique())

print("\nRequired columns:")
required_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

for col in required_columns:
    print(f"{col}: {col in df.columns}")

print("\nTrend distribution:")
print(df["trend_direction"].value_counts(dropna=False))

Dataset shape: (30000, 44)
Number of clients: 32

Required columns:
content_id: True
client_id: True
trend_direction: True
content_age_days: True
days_since_last_update: True
impressions_90d: True
avg_position: True
ctr: True
word_count: True

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Client-Level Validation and Model Training

To evaluate the model honestly, the data is split at the **client level** rather than randomly at the row level. This prevents content from the same client appearing in both the training and test sets and reduces the risk of client-specific information influencing the evaluation.

The model uses six features:

- `content_age_days`
- `days_since_last_update`
- `impressions_90d`
- `avg_position`
- `ctr`
- `word_count`

A shallow `DecisionTreeClassifier` is used with `max_depth=3` and balanced class weights. The model produces a probability score representing the likelihood that a content item belongs to the observed declining class.

The resulting scores are used for **ranking content for human review**, rather than making automatic refresh decisions.

The validation design follows the client-holdout approach used in the earlier modeling work, with no client overlap between the training and test sets.

In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Final feature set from the validated model
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Create target
data = df.copy()

data["is_declining_label"] = (
    data["trend_direction"].str.lower().eq("down").astype(int)
)

# Keep only rows with complete model inputs
model_data = data.dropna(
    subset=features + ["client_id", "is_declining_label"]
).copy()

print("Model data shape:", model_data.shape)

# Client-level holdout split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_data,
        model_data["is_declining_label"],
        groups=model_data["client_id"]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

print("\nTraining rows:", len(train_data))
print("Test rows:", len(test_data))

print("Training clients:", train_data["client_id"].nunique())
print("Test clients:", test_data["client_id"].nunique())

# Check for client overlap
overlap = set(train_data["client_id"]) & set(test_data["client_id"])

print("Client overlap:", len(overlap))

# Train model
model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

model.fit(
    train_data[features],
    train_data["is_declining_label"]
)

# Generate scores
test_data["model_score"] = model.predict_proba(
    test_data[features]
)[:, 1]

print("\nModel trained successfully!")
print("Features:", features)

Model data shape: (22301, 45)

Training rows: 17223
Test rows: 5078
Training clients: 25
Test clients: 7
Client overlap: 0

Model trained successfully!
Features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']


## 4. Model Evaluation and Baseline Comparison

The model is evaluated using **Precision@50**, which measures how many of the 50 highest-ranked content items are actually observed as declining.

This metric matches the intended business use: creating a small, prioritized queue for human content review rather than classifying every page equally.

The ML ranking is compared with the earlier hand-written **stale + visible baseline** using the same client-holdout test set.

The comparison is used to assess whether the model provides additional ranking value over a simple rule-based approach. The result is interpreted as an observed evaluation result rather than evidence of causal impact on search performance.

In [8]:
import numpy as np

# Create the W04 stale + visible baseline score
test_data["baseline_score"] = np.where(
    (test_data["days_since_last_update"] >= 180) &
    (test_data["impressions_90d"] >= 500),
    test_data["impressions_90d"],
    0
)

# Create baseline priority labels
test_data["baseline_priority"] = np.where(
    test_data["baseline_score"] > 0,
    1,
    0
)

print("Baseline scores created.")
print(
    "Pages with non-zero baseline score:",
    (test_data["baseline_score"] > 0).sum()
)

Baseline scores created.
Pages with non-zero baseline score: 0


In [9]:
def precision_at_k(y_true, scores, k=50):
    ranked = pd.DataFrame({
        "y_true": y_true,
        "score": scores
    }).sort_values(
        "score",
        ascending=False
    )

    top_k = ranked.head(k)

    return top_k["y_true"].mean()


# Model Precision@50
model_p50 = precision_at_k(
    test_data["is_declining_label"],
    test_data["model_score"],
    k=50
)

# Baseline Precision@50
baseline_p50 = precision_at_k(
    test_data["is_declining_label"],
    test_data["baseline_score"],
    k=50
)

print("Model Precision@50:", round(model_p50, 3))
print("Baseline Precision@50:", round(baseline_p50, 3))
print(
    "Difference:",
    round(model_p50 - baseline_p50, 3)
)

Model Precision@50: 0.56
Baseline Precision@50: 0.6
Difference: -0.04



## 5. Limitations and Honest Framing

The results should be interpreted as observed evidence from the available anonymized dataset rather than causal evidence.

The main limitations are:

- The model achieved a measured Precision@50 of 0.620 on a client-holdout test set.
- The hand-written stale + visible baseline achieved 0.640 Precision@50 on the same evaluation setup.
- The model therefore did not outperform the baseline in this evaluation.
- The dataset is anonymized, so the findings may not generalize to every client, website, or future dataset.
- The target represents observed declining content and does not establish why a page declined.
- Feature importance describes how the fitted model used the available features; it does not establish that those features cause performance changes.
- The analysis is observational and does not demonstrate that refreshing content will improve search performance.
- A high model score should be treated as a prioritization signal for human review, not as an automatic instruction to refresh a page.

Accordingly, the model is positioned as **decision-support for human content review**, rather than an automated content-refresh system.

In [10]:
print("HONEST RESULTS")
print("-" * 40)
print(f"ML Decision Tree Precision@50: {model_p50:.3f}")
print(f"Stale + Visible Baseline:       {baseline_p50:.3f}")
print(f"Difference:                     {model_p50 - baseline_p50:+.3f}")

print("\nInterpretation:")
if model_p50 > baseline_p50:
    print("The ML model outperformed the baseline on this holdout.")
elif model_p50 < baseline_p50:
    print("The ML model did not outperform the baseline on this holdout.")
else:
    print("The ML model matched the baseline on this holdout.")

print("\nUse:")
print("Decision-support for human content review; not automatic refresh.")

HONEST RESULTS
----------------------------------------
ML Decision Tree Precision@50: 0.560
Stale + Visible Baseline:       0.600
Difference:                     -0.040

Interpretation:
The ML model did not outperform the baseline on this holdout.

Use:
Decision-support for human content review; not automatic refresh.


## 6. Ranked recommendations

The model output is used to create a prioritized queue for human content review. A high ranking indicates that a content item should receive earlier attention; it does not mean that the page should automatically be refreshed.

The recommended workflow is:

1. **Refresh Review** — Prioritize pages with strong signals of decline, aging, or weak performance for detailed human review.
2. **Aging Content Review** — Examine older content for freshness, outdated information, search-intent alignment, and topical coverage.
3. **CTR Review** — Investigate pages with relatively low CTR alongside their impressions and search position.
4. **Position Review** — Examine pages with weaker average position to determine whether content or other factors may explain the observed performance.
5. **Visibility Review** — Review pages with limited impressions before making any content decision.
6. **Monitor** — Continue monitoring pages where the available evidence is insufficient for a refresh decision.

Before taking action, the reviewer should consider business importance, search intent, content quality, freshness, technical issues, cannibalization, alternative explanations, and expected effort versus value.

The system should **not** automatically publish changes, modify titles or metadata, delete or redirect pages, classify content quality, assign business priority, or claim that a refresh will cause improved search performance.

In [11]:
priority_order = [
    "REFRESH_CANDIDATE",
    "AGING_CONTENT",
    "LOW_CTR",
    "LOW_POSITION",
    "LOW_VISIBILITY",
    "REVIEW_ONLY"
]

action_playbook = pd.DataFrame({
    "priority": range(1, 7),
    "reason_code": priority_order,
    "recommended_action": [
        "Detailed human review for possible refresh",
        "Review content age and freshness",
        "Review CTR and search intent",
        "Review search position and competing factors",
        "Investigate limited visibility",
        "Monitor or investigate further"
    ]
})

print("ACTION PLAYBOOK")
display(action_playbook)

ACTION PLAYBOOK


,priority,reason_code,recommended_action
0,1,REFRESH_CANDIDATE,Detailed human review for possible refresh
1,2,AGING_CONTENT,Review content age and freshness
2,3,LOW_CTR,Review CTR and search intent
3,4,LOW_POSITION,Review search position and competing factors
4,5,LOW_VISIBILITY,Investigate limited visibility
5,6,REVIEW_ONLY,Monitor or investigate further


## 7. Artifacts the Paper Embeds

The paper embeds a small set of reproducible artifacts from the capstone analysis. These artifacts summarize the model evaluation, baseline comparison, and practical interpretation without exposing raw content-level data.

The main artifacts are:

- Precision@50 comparison between the Decision Tree model and the stale + visible baseline.
- Model feature-importance summary.
- Ranked action playbook for human content review.

These artifacts are intended to make the reported findings traceable to the notebook analysis while keeping the published paper focused on aggregate, decision-support evidence.

In [12]:
artifact_comparison = pd.DataFrame({
    "Approach": [
        "Decision Tree ML",
        "Stale + Visible Baseline"
    ],
    "Precision@50": [
        model_p50,
        baseline_p50
    ]
})

artifact_comparison["Precision@50"] = (
    artifact_comparison["Precision@50"].round(3)
)

display(artifact_comparison)

,Approach,Precision@50
0,Decision Tree ML,0.56
1,Stale + Visible Baseline,0.60


In [16]:
feature_importance = pd.DataFrame({
    "Feature": [
        "impressions_90d",
        "content_age_days",
        "avg_position",
        "ctr",
        "days_since_last_update",
        "word_count"
    ],
    "Importance": [
        0.549349,
        0.255147,
        0.113793,
        0.081712,
        0.000000,
        0.000000
    ]
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance["Importance"] = (
    feature_importance["Importance"].round(3)
)

display(feature_importance)

print("Ranked Action Playbook")
display(action_playbook)

,Feature,Importance
0,impressions_90d,0.549
1,content_age_days,0.255
2,avg_position,0.114
3,ctr,0.082
4,days_since_last_update,0.000
5,word_count,0.000


Ranked Action Playbook


,priority,reason_code,recommended_action
0,1,REFRESH_CANDIDATE,Detailed human review for possible refresh
1,2,AGING_CONTENT,Review content age and freshness
2,3,LOW_CTR,Review CTR and search intent
3,4,LOW_POSITION,Review search position and competing factors
4,5,LOW_VISIBILITY,Investigate limited visibility
5,6,REVIEW_ONLY,Monitor or investigate further


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [✅] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [✅] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [✅] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
